In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('telo_cleaned.csv')
display(df.head())

,SeniorCitizen,Partner,Dependents,tenure,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Yes,No,1,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,0,No,No,34,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,0,No,No,2,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,0,No,No,45,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,0,No,No,2,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})
cat_cols = df.select_dtypes(include=['object']).columns
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print("Khích thước dữ liệu sau mã hóa:", df_encoded.shape)
display(df_encoded.head())

Khích thước dữ liệu sau mã hóa: (7032, 29)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,Partner_Yes,Dependents_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,True,False,True,False,False,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,False,False,True,False,False,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,False,True,...,False,False,False,False,False,False,True,False,True,False


In [5]:
X = df_encoded.drop("Churn", axis = 1)
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state=42, stratify=y)
print(f"số lượng mẫu train:{X_train.shape[0]}")
print(f"Số lượng mẫu test {X_test.shape[0]}")

số lượng mẫu train:5625
Số lượng mẫu test 1407


Chuẩn hóa dữ liệu 

In [8]:
scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])



xử lý mất cân bằng dữ liệu (SMOTE)

Chọn điểm A trong bộ Churn = 1

chọn ngẫu nhiên 1 trong k điểm gần nhất gọi là B, sau 

Tạo ra điểm mới là C ở giữa A và B

$$
C = A + \lambda \times (B - A)
$$

In [25]:
def manual_smote(X_minority, num_synthetic_samples, k=5):
    # Thêm dòng này ở đầu hàm manual_smote
    X_minority = np.array(X_minority, dtype=np.float64)
    n_samples, n_features = X_minority.shape
    synthetic_data = []

    for _ in range(num_synthetic_samples):
        idx_A = np.random.randint(0, n_samples)
        A = X_minority[idx_A]

        distances = np.linalg.norm(X_minority-A, axis = 1)
        nearest_indices = np.argsort(distances)[1:k+1]

        idx_B = np.random.choice(nearest_indices)
        B = X_minority[idx_B]

        lambd = np.random.rand()
        C = A + lambd*(B-A)

        synthetic_data.append(C)

    return np.array(synthetic_data)


In [28]:
X_train_np = X_train.values
y_train_np = y_train.values

X_majority = X_train_np[y_train_np ==0]
X_minority = X_train_np[y_train_np ==1]

num_to_gen = len(X_majority) - len(X_minority)
X_synthetic = manual_smote(X_minority, num_to_gen, k=5)
y_synthetic = np.ones(num_to_gen)
X_train_resampled_np = np.vstack((X_majority,X_minority,X_synthetic))
y_train_resampled_np = np.hstack((np.zeros(len(X_majority)), np.ones(len(X_minority)), y_synthetic))

X_train_resampled = pd.DataFrame(X_train_resampled_np, columns=X_train.columns)
y_train_resampled = pd.Series(y_train_resampled_np, name='Churn')

print("Số lượng mẫu sau khi dùng SMOTE:", y_train_resampled.value_counts())

Số lượng mẫu sau khi dùng SMOTE: Churn
0.0    4130
1.0    4130
Name: count, dtype: int64


In [32]:
display(X_train_resampled.head())

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Partner_Yes,Dependents_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1.321816,0.981556,1.6599,True,True,False,True,True,False,...,False,False,False,False,False,True,False,True,False,False
1,0,-0.26741,-0.971546,-0.562252,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,0,1.444064,0.837066,1.756104,True,False,False,True,True,False,...,False,False,False,False,False,True,False,True,False,False
3,0,-1.204646,0.641092,-0.908326,False,False,False,False,True,False,...,False,False,False,True,False,False,False,False,True,False
4,0,0.669826,-0.808787,-0.101561,True,False,True,False,False,False,...,False,True,False,False,False,False,False,False,False,False


In [4]:
import CustomLogisticRegression

model_standard = CustomLogisticRegression(learning_rate = 0.01, num_iterations = 2000, penalty = 'none')
model_ridge = CustomLogisticRegression(learning_rate=0.01, num_iterations=2000, penalty='l2', lambda_param=1.0)
model_lasso = CustomLogisticRegression(learning_rate=0.01, num_iterations=2000, penalty='l1', lambda_param=1.0)



TypeError: 'module' object is not callable

In [ ]:
# Chọn 1 mô hình để huấn luyện (Ví dụ: Lasso)
print("Đang huấn luyện mô hình Lasso...")
model_lasso.fit(X_train_resampled_np, y_train_resampled_np)
predictions = model_lasso.predict(X_test.values)
print("Huấn luyện xong!")